
# MicrobeID: Multimodal Machine Learning for Bacterial Species Classification

**Project goal:** Investigate whether combining bacterial colony images with agar-medium information improves bacterial species classification compared with using images or medium information alone.

## Research Question

Can combining bacterial colony morphology with growth-medium characteristics improve machine learning classification of bacterial species compared with using colony images alone?

## Classification Models

1. **Image-only:** Uses colony images to extract visual features with pretrained MobileNetV2.
2. **Medium-only:** Uses agar-medium information as the classification input.
3. **Combined:** Uses both extracted image features and agar-medium information.

## Dataset

The project uses publicly available bacterial colony images from two Mendeley Data repositories. The main analysis focuses on:

- *Escherichia coli*
- *Klebsiella pneumoniae*
- *Pseudomonas aeruginosa*

The species were represented on MacConkey and brain heart infusion (BHI) agar.

## Project Status

Dataset organization, duplicate screening, and plate-level training/testing assignment have been completed. Image preprocessing, feature extraction, model training, and evaluation remain to be completed.



## 1. Dataset Sources and Description

Two publicly available datasets from Mendeley Data were used for this project.

### Dataset 1: MacConkey Agar

Nagro et al. (2022) provided bacterial colony images of *Escherichia coli* and *Klebsiella pneumoniae* grown on MacConkey agar.

- Dataset: https://doi.org/10.17632/kx6gz3wmcf.1
- Associated publication: https://doi.org/10.1109/ACCESS.2022.3221958

### Dataset 2: BHI and MacConkey Agar

Kutbi et al. (2025) provided images of multiple bacterial species grown on brain heart infusion (BHI) agar, along with additional *Pseudomonas aeruginosa* images grown on MacConkey agar.

- Dataset: https://doi.org/10.17632/v54x8jdx5x.1
- Associated publication: https://doi.org/10.1109/ACCESS.2025.3625648

### Dataset Organization

The original image collection contained 22,167 images. After duplicate screening and exclusion of images with uncertain plate assignments, 22,157 images remained.

The main classification analysis focuses on three bacterial species represented on both growth media:

- *E. coli*
- *K. pneumoniae*
- *P. aeruginosa*

*Staphylococcus aureus* was excluded from the main comparison because it was only represented on BHI agar.

The final three-species dataset contains 18,432 images.


In [19]:
from pathlib import Path
import pandas as pd

backup = Path("/content/drive/MyDrive/MicrobeID_Backup")

original = pd.read_csv(backup / "original_manifest.csv")
cleaned = pd.read_csv(backup / "cleaned_model_manifest.csv")
main = pd.read_csv(backup / "main_manifest.csv")
training = pd.read_csv(backup / "training_manifest.csv")
testing = pd.read_csv(backup / "testing_manifest.csv")

print("Original:", original.shape)
print("Cleaned:", cleaned.shape)
print("Main:", main.shape)
print("Training:", training.shape)
print("Testing:", testing.shape)

Original: (22167, 9)
Cleaned: (22157, 9)
Main: (18432, 12)
Training: (14942, 12)
Testing: (3490, 12)


In [1]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:

import os

backup_path = "/content/drive/MyDrive/MicrobeID_Backup"

print("Folder exists:", os.path.exists(backup_path))

if os.path.exists(backup_path):
    print("\nFiles in MicrobeID_Backup:")
    for filename in sorted(os.listdir(backup_path)):
        print(filename)


Folder exists: True

Files in MicrobeID_Backup:
cleaned_model_manifest.csv
main_manifest.csv
original_manifest.csv
testing_manifest.csv
training_manifest.csv


In [10]:

import os
import requests
import zipfile
from pathlib import Path

# Create a dataset directory in the Colab runtime
base_dir = Path("/content/MicrobeID_Datasets")
base_dir.mkdir(parents=True, exist_ok=True)

# Direct public download links supplied for the project
dataset_links = {
    "2022_archive_1": "https://data.mendeley.com/public-files/datasets/kx6gz3wmcf/files/25ee2fb0-a330-4dcb-908b-53ccd14047f7/file_downloaded",
    "2022_archive_2": "https://data.mendeley.com/public-files/datasets/kx6gz3wmcf/files/21809afb-2a1a-43d7-96f3-bd92117f9fd1/file_downloaded",
    "2022_archive_3": "https://data.mendeley.com/public-files/datasets/kx6gz3wmcf/files/55380019-f71b-4217-99af-b02d2236e89f/file_downloaded",
    "2022_archive_4": "https://data.mendeley.com/public-files/datasets/kx6gz3wmcf/files/efd8394d-9d64-4d59-b561-0871c66d424b/file_downloaded",
    "2025_archive_1": "https://data.mendeley.com/public-files/datasets/v54x8jdx5x/files/e27a754b-6421-4c54-9b63-ccf41ce7693f/file_downloaded",
    "2025_archive_2": "https://data.mendeley.com/public-files/datasets/v54x8jdx5x/files/1561f231-784e-4445-b800-78b89bf84681/file_downloaded"
}

# Download and extract each archive
image_extensions = (".jpg", ".jpeg", ".heic", ".png")

def safe_extract(zip_file, destination):
    """Extract ZIP contents while preventing unsafe file paths."""
    destination = destination.resolve()

    with zipfile.ZipFile(zip_file, "r") as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()

            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe archive path: {member.filename}")

        archive.extractall(destination)

total_images = 0

for archive_name, url in dataset_links.items():

    print(f"\nDownloading {archive_name}...")

    zip_path = base_dir / f"{archive_name}.zip"
    extract_path = base_dir / archive_name
    extract_path.mkdir(exist_ok=True)

    response = requests.get(url, stream=True, timeout=180)
    response.raise_for_status()

    with open(zip_path, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

    if not zipfile.is_zipfile(zip_path):
        raise ValueError(
            f"{archive_name} did not download as a valid ZIP file."
        )

    print("Download complete. Extracting...")

    safe_extract(zip_path, extract_path)

    # Count extracted images
    image_files = [
        path for path in extract_path.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    ]

    image_count = len(image_files)
    total_images += image_count

    print(f"Images extracted: {image_count:,}")

    # Remove ZIP after successful extraction to save space
    zip_path.unlink()

print("\n" + "=" * 45)
print("ALL DATASET ARCHIVES EXTRACTED")
print("=" * 45)
print(f"Total images extracted: {total_images:,}")

print("\nExtracted folders:")
for folder in sorted(base_dir.iterdir()):
    if folder.is_dir():
        print(folder.name)



Download complete. Extracting...
Images extracted: 1,170

Download complete. Extracting...
Images extracted: 1,811

Download complete. Extracting...
Images extracted: 2,530

Download complete. Extracting...
Images extracted: 1,854

Download complete. Extracting...
Images extracted: 2,454

Download complete. Extracting...
Images extracted: 0

ALL DATASET ARCHIVES EXTRACTED
Total images extracted: 9,819

Extracted folders:
2022
2022_archive_1
2022_archive_2
2022_archive_3
2022_archive_4
2025_archive_1
2025_archive_2


In [11]:
from pathlib import Path
from collections import Counter

folder = Path("/content/MicrobeID_Datasets/2025_archive_2")

print("Folder exists:", folder.exists())

files = [f for f in folder.rglob("*") if f.is_file()]

print("Total files found:", len(files))

print("\nFirst 20 files:")
for f in files[:20]:
    print(f.relative_to(folder))

print("\nFile extensions:")
extensions = Counter(f.suffix.lower() for f in files)
for ext, count in extensions.items():
    print(ext if ext else "[no extension]", count)

Folder exists: True
Total files found: 4

First 20 files:
Staph On BH.zip
e.coli on BH.zip
Pseudomonas aeruginosa on BH.zip
Klebsiella on BH.zip

File extensions:
.zip 4


In [12]:
from pathlib import Path
import zipfile

parent_folder = Path("/content/MicrobeID_Datasets/2025_archive_2")

total_images = 0
image_extensions = {".jpg", ".jpeg", ".png", ".heic"}

for zip_path in parent_folder.glob("*.zip"):

    extract_folder = parent_folder / zip_path.stem
    extract_folder.mkdir(exist_ok=True)

    print(f"\nExtracting {zip_path.name}...")

    with zipfile.ZipFile(zip_path, "r") as archive:
        bad_file = archive.testzip()

        if bad_file:
            print("Problem found in archive:", bad_file)
            continue

        archive.extractall(extract_folder)

    image_count = sum(
        1 for file in extract_folder.rglob("*")
        if file.is_file() and file.suffix.lower() in image_extensions
    )

    total_images += image_count
    print("Images extracted:", image_count)

print("\n===================================")
print("NESTED ARCHIVES EXTRACTED")
print("===================================")
print("Total images extracted:", total_images)


Extracting Staph On BH.zip...
Images extracted: 3726

Extracting e.coli on BH.zip...
Images extracted: 3061

Extracting Pseudomonas aeruginosa on BH.zip...
Images extracted: 2824

Extracting Klebsiella on BH.zip...
Images extracted: 2737

NESTED ARCHIVES EXTRACTED
Total images extracted: 12348


In [13]:
from pathlib import Path
from collections import Counter

root = Path("/content/MicrobeID_Datasets")

image_extensions = {".jpg", ".jpeg", ".png", ".heic"}

for folder in sorted(root.iterdir()):
    if folder.is_dir():
        images = [
            f for f in folder.rglob("*")
            if f.is_file() and f.suffix.lower() in image_extensions
        ]

        if images:
            print(f"{folder.name}: {len(images)} images")

2022_archive_1: 1170 images
2022_archive_2: 1811 images
2022_archive_3: 2530 images
2022_archive_4: 1854 images
2025_archive_1: 2454 images
2025_archive_2: 12348 images


In [14]:
from pathlib import Path

root = Path("/content/MicrobeID_Datasets")

for archive_folder in sorted(root.iterdir()):
    if archive_folder.is_dir():
        print(f"\n{archive_folder.name}")

        for item in sorted(archive_folder.iterdir()):
            if item.is_dir():
                print(f"  {item.name}")


2022

2022_archive_1
  E. Coli

2022_archive_2
  K. pneumoniae

2022_archive_3
  e.coli uncontrolled

2022_archive_4
  k.pneumoniae uncomtrolled

2025_archive_1
  Group3

2025_archive_2
  Klebsiella on BH
  Pseudomonas aeruginosa on BH
  Staph On BH
  e.coli on BH


In [16]:
import pandas as pd

backup = "/content/drive/MyDrive/MicrobeID_Backup"

original = pd.read_csv(f"{backup}/original_manifest.csv")

print("Manifest shape:", original.shape)
print("\nColumns:")
print(original.columns.tolist())

print("\nSample image paths:")
print(original["image_path"].head(10).to_string(index=False))

Manifest shape: (22167, 9)

Columns:
['image_path', 'filename', 'species', 'medium', 'condition', 'archive', 'plate_id', 'extension', 'sha256']

Sample image paths:
/content/2022_Group1/E. Coli/Uncontrolled/Plate...
/content/2022_Group1/E. Coli/Uncontrolled/Plate...
/content/2022_Group1/E. Coli/Uncontrolled/Plate...
/content/2022_Group1/E. Coli/Uncontrolled/Plate...
/content/2022_Group1/E. Coli/Uncontrolled/Plate...
/content/2022_Group1/E. Coli/Uncontrolled/Plate...
/content/2022_Group1/E. Coli/Uncontrolled/Plate...
/content/2022_Group1/E. Coli/Uncontrolled/Plate...
/content/2022_Group1/E. Coli/Uncontrolled/Plate...
/content/2022_Group1/E. Coli/Uncontrolled/Plate...


In [17]:
from pathlib import Path

print("OLD MANIFEST PATHS")
for path in original["image_path"].head(5):
    print(path)

print("\nNEW DATASET PATH EXAMPLES")
root = Path("/content/MicrobeID_Datasets")

for folder in sorted(root.iterdir()):
    if folder.is_dir():
        examples = [
            p for p in folder.rglob("*")
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".heic"}
        ]

        print(f"\n{folder.name}:")
        for path in examples[:3]:
            print(path)

OLD MANIFEST PATHS
/content/2022_Group1/E. Coli/Uncontrolled/Plate 22/6f.JPG
/content/2022_Group1/E. Coli/Uncontrolled/Plate 22/30d.JPG
/content/2022_Group1/E. Coli/Uncontrolled/Plate 22/IMG_6553.HEIC
/content/2022_Group1/E. Coli/Uncontrolled/Plate 22/IMG_6712.HEIC
/content/2022_Group1/E. Coli/Uncontrolled/Plate 22/IMG_6551.HEIC

NEW DATASET PATH EXAMPLES

2022:

2022_archive_1:
/content/MicrobeID_Datasets/2022_archive_1/E. Coli/Controlled/Plate 31/IMG_8741.JPG
/content/MicrobeID_Datasets/2022_archive_1/E. Coli/Controlled/Plate 31/IMG_8740.JPG
/content/MicrobeID_Datasets/2022_archive_1/E. Coli/Controlled/Plate 23/IMG_8703.JPG

2022_archive_2:
/content/MicrobeID_Datasets/2022_archive_2/K. pneumoniae/Controlled/Plate 31/Photo 25-04-2021, 11 04 58 PM.jpg
/content/MicrobeID_Datasets/2022_archive_2/K. pneumoniae/Controlled/Plate 31/Photo 25-04-2021, 11 05 34 PM.jpg
/content/MicrobeID_Datasets/2022_archive_2/K. pneumoniae/Controlled/Plate 31/Photo 25-04-2021, 11 04 30 PM.jpg

2022_archive_3:

In [18]:
from pathlib import Path
import hashlib
from collections import defaultdict

root = Path("/content/MicrobeID_Datasets")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

image_extensions = {".jpg", ".jpeg", ".png", ".heic"}

new_hashes = defaultdict(list)

for path in root.rglob("*"):
    if path.is_file() and path.suffix.lower() in image_extensions:
        new_hashes[sha256_file(path)].append(str(path))

matched = original["sha256"].isin(new_hashes)

print("Original manifest images:", len(original))
print("Matched by SHA-256:", matched.sum())
print("Unmatched:", (~matched).sum())

Original manifest images: 22167
Matched by SHA-256: 22167
Unmatched: 0


In [20]:
# Create a mapping from image hash to its new file path

hash_to_new_path = {}

for path in root.rglob("*"):
    if path.is_file() and path.suffix.lower() in image_extensions:
        file_hash = sha256_file(path)
        hash_to_new_path[file_hash] = str(path)

# Check that every original image has a matching new path
original_hashes = set(original["sha256"])
matched_hashes = original_hashes.intersection(hash_to_new_path)

print("Original unique hashes:", len(original_hashes))
print("Matched unique hashes:", len(matched_hashes))
print("Missing hashes:", len(original_hashes - matched_hashes))

Original unique hashes: 22159
Matched unique hashes: 22159
Missing hashes: 0


In [3]:

import pandas as pd

# Load the saved manifests
cleaned_manifest = pd.read_csv(
    os.path.join(backup_path, "cleaned_model_manifest.csv")
)

main_manifest = pd.read_csv(
    os.path.join(backup_path, "main_manifest.csv")
)

original_manifest = pd.read_csv(
    os.path.join(backup_path, "original_manifest.csv")
)

training_manifest = pd.read_csv(
    os.path.join(backup_path, "training_manifest.csv")
)

testing_manifest = pd.read_csv(
    os.path.join(backup_path, "testing_manifest.csv")
)

# Display dataset sizes
print("Original images:", len(original_manifest))
print("Cleaned images:", len(cleaned_manifest))
print("Main comparison images:", len(main_manifest))
print("Training images:", len(training_manifest))
print("Testing images:", len(testing_manifest))


Original images: 22167
Cleaned images: 22157
Main comparison images: 18432
Training images: 14942
Testing images: 3490


In [4]:

print("MAIN MANIFEST COLUMNS:")
print(main_manifest.columns.tolist())

print("\nFIRST FIVE ROWS:")
display(main_manifest.head())

print("\nTRAINING MANIFEST COLUMNS:")
print(training_manifest.columns.tolist())

print("\nTESTING MANIFEST COLUMNS:")
print(testing_manifest.columns.tolist())


MAIN MANIFEST COLUMNS:
['image_path', 'filename', 'species', 'medium', 'condition', 'archive', 'plate_id', 'extension', 'sha256', 'plate_number', 'split_group', 'stratify_label']

FIRST FIVE ROWS:


,image_path,filename,species,medium,condition,archive,plate_id,extension,sha256,plate_number,split_group,stratify_label
0,/content/2022_Group1/E. Coli/Uncontrolled/Plat...,6f.JPG,E. coli,MacConkey,Uncontrolled,2022_Ecoli_Group1,Plate 22,.jpg,2e8b82e3f35d1b3ebc05e2a421fb19b13cb17440d6b907...,22,2022_Ecoli_Group1_E. coli_Plate22,E. coli_MacConkey
1,/content/2022_Group1/E. Coli/Uncontrolled/Plat...,30d.JPG,E. coli,MacConkey,Uncontrolled,2022_Ecoli_Group1,Plate 22,.jpg,17576b5e72eb3852c9d3844189695aad8a4ab8cb7bfc8b...,22,2022_Ecoli_Group1_E. coli_Plate22,E. coli_MacConkey
2,/content/2022_Group1/E. Coli/Uncontrolled/Plat...,IMG_6553.HEIC,E. coli,MacConkey,Uncontrolled,2022_Ecoli_Group1,Plate 22,.heic,976e348b929910bcbfa0899f074ed7bebe0cd86f2cba87...,22,2022_Ecoli_Group1_E. coli_Plate22,E. coli_MacConkey
3,/content/2022_Group1/E. Coli/Uncontrolled/Plat...,IMG_6712.HEIC,E. coli,MacConkey,Uncontrolled,2022_Ecoli_Group1,Plate 22,.heic,3cf6848012b5366623b3880596d3d26cd950f58cf78b5b...,22,2022_Ecoli_Group1_E. coli_Plate22,E. coli_MacConkey
4,/content/2022_Group1/E. Coli/Uncontrolled/Plat...,IMG_6551.HEIC,E. coli,MacConkey,Uncontrolled,2022_Ecoli_Group1,Plate 22,.heic,1169936785c2f12f35fd72ddc755acb63e73e9d3d5f038...,22,2022_Ecoli_Group1_E. coli_Plate22,E. coli_MacConkey



TRAINING MANIFEST COLUMNS:
['image_path', 'filename', 'species', 'medium', 'condition', 'archive', 'plate_id', 'extension', 'sha256', 'plate_number', 'split_group', 'stratify_label']

TESTING MANIFEST COLUMNS:
['image_path', 'filename', 'species', 'medium', 'condition', 'archive', 'plate_id', 'extension', 'sha256', 'plate_number', 'split_group', 'stratify_label']



## 2. Data Cleaning and Train/Test Split

### Duplicate Screening

Exact duplicate images were identified using SHA-256 file hashes. Images with duplicate content were reviewed to determine whether they belonged to the same plate or had conflicting plate assignments.

A total of 10 images were excluded:
- Duplicate copies from the same plate.
- Images associated with conflicting plate assignments that could not be confidently resolved.

After screening, the cleaned dataset contained 22,157 images. The main three-species dataset contained 18,432 images.

### Training and Testing Assignment

The main dataset was divided into training and testing sets using StratifiedGroupKFold with five folds, shuffling, and a random seed of 42.

Plate-level grouping was used to prevent images from the same plate group from appearing in both the training and testing sets. This helps reduce data leakage and provides a more reliable evaluation of model performance.

The final split was:

| Dataset | Images | Plate Groups |
|---|---:|---:|
| Training | 14,942 | 175 |
| Testing | 3,490 | 43 |
| **Total** | **18,432** | **218** |

No plate groups overlapped between the training and testing sets.

The same training and testing assignments will be used for all three classification models to ensure a consistent comparison.



## 3. Model Development and Evaluation

Three machine learning models will be developed to investigate whether growth-medium information improves bacterial species classification.

### Model 1: Image-Only

Colony images will be processed and resized to 224 × 224 pixels. A pretrained MobileNetV2 convolutional neural network will be used to extract visual features from each image. These features will then be classified using logistic regression.

### Model 2: Medium-Only

Growth-medium information (MacConkey or BHI agar) will be converted into numerical features and used as the sole input for logistic regression. This model will establish a baseline for determining how well growth-medium information alone predicts bacterial species.

### Model 3: Combined Image and Medium

The visual features extracted using MobileNetV2 will be combined with the numerical growth-medium information. Logistic regression will then classify bacterial species using both inputs.

### Model Evaluation

All three models will use the same training and testing assignments. Performance will be evaluated using:

- Accuracy
- Precision
- Recall
- Macro-averaged F1 score

Confusion matrices will also be generated to examine classification errors across the three bacterial species.

The results will be compared to determine whether combining colony morphology and growth-medium information improves classification compared with either input alone.



## 4. Preliminary Results

### Dataset Summary

The original dataset contained 22,167 images. After duplicate screening and exclusion of uncertain plate assignments, 22,157 images remained.

The main three-species comparison contained 18,432 images.

| Bacterial Species | BHI | MacConkey | Total |
|---|---:|---:|---:|
| *E. coli* | 3,061 | 3,700 | 6,761 |
| *K. pneumoniae* | 2,735 | 3,661 | 6,396 |
| *P. aeruginosa* | 2,824 | 2,451 | 5,275 |
| **Total** | **8,620** | **9,812** | **18,432** |

### Training and Testing Summary

The final dataset was divided into 14,942 training images and 3,490 testing images. The 218 plate groups were separated into 175 training groups and 43 testing groups, with no overlapping groups.

### Remaining Results

The following analyses have not yet been completed:

- MobileNetV2 image feature extraction.
- Training the image-only, medium-only, and combined models.
- Comparing accuracy, precision, recall, and macro F1 scores.
- Generating confusion matrices.
- Interpreting differences in model performance.

**These results and visualizations will be added after model training and evaluation are completed.**



## 5. Reproducibility, Limitations, and Next Steps

### Reproducibility

The project is being developed in Python using Google Colab. Dataset organization, duplicate screening, and training/testing assignments were performed programmatically.

A random seed of 42 was used for the plate-level split. The same split will be maintained across all three classification models to support a fair comparison.

The original datasets are publicly available through Mendeley Data. Dataset links and associated publications are provided above.

### Limitations

Several limitations may influence model performance:

- The images were collected under both controlled and uncontrolled imaging conditions, resulting in variation in lighting, camera angle, and image quality.
- The number of images differs across bacterial species and growth media.
- The *P. aeruginosa* BHI testing set contains only one independent plate group, limiting the ability to evaluate generalization across different plates.
- The datasets were collected from separate studies, which may introduce differences in image acquisition and data organization.

### Next Steps

The remaining project work includes:

1. Preprocessing images into a consistent RGB format and resizing them to 224 × 224 pixels.
2. Extracting image features using pretrained MobileNetV2.
3. Training and evaluating all three classification models.
4. Generating model comparison tables and confusion matrices.
5. Interpreting the results and completing the final report.
